# Análise Exploratória de Dados
`passo-a-passo`

In [1]:
!pip uninstall sql_blocks
!pip install sql_blocks --upgrade   #--- Baixando a biblioteca para o nosso ambiente

Found existing installation: sql_blocks 1.20260419
Uninstalling sql_blocks-1.20260419:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/sql_blocks-1.20260419.dist-info/*
    /usr/local/lib/python3.12/dist-packages/sql_blocks/*
Proceed (Y/n)? 
  Successfully uninstalled sql_blocks-1.20260419
  Using cached sql_blocks-1.20260419-py3-none-any.whl.metadata (28 kB)
Using cached sql_blocks-1.20260419-py3-none-any.whl (46 kB)


In [2]:
import duckdb

TABELA_CLIENTES = "'sample_data/clientes.csv' c"
TABELA_VENDAS   = "'sample_data/vendas.csv'   v"
TABELA_PRODUTOS = "'sample_data/produtos.csv' p"

### Comece listando as funções disponíveis

In [3]:
from sql_blocks import Function

Function.list_all()  # <-----------------------------

Aggregate Function Avg(FLOAT)  Return FLOAT
     Calculate the average (arithmetic mean) of a 
    set of numeric values within a specified column
    
--------------------
Aggregate Function Count(FLOAT)  Return FLOAT
    Return the number of rows that matches a specified criterion.
    
--------------------
Aggregate Function Max(FLOAT)  Return FLOAT
    Returns the largest value of the selected column.
    
--------------------
Aggregate Function Min(FLOAT)  Return FLOAT
    Returns the smallest value of the selected column.
    
--------------------
Aggregate Function StdDev(FLOAT)  Return FLOAT
    Calculates the sample standard deviator of values
    
--------------------
Aggregate Function Sum(FLOAT)  Return FLOAT
--------------------
Date Function Day(DATE)  Return INT
--------------------
Date Function Month(DATE)  Return INT
--------------------
Date Function Year(DATE)  Return INT
--------------------
Function Cast(ANY As ANY)  Return ANY
    Converts a value (of any type) i

### Escolha as funções que vai usar
> Não se esqueça de incluir **Select** e **EDA**! 😉

In [4]:
from sql_blocks import Select, EDA, Min, Max, Count

### Monte sua primeira consulta dando nomes a cada campo calculado...

In [5]:
query = Select(
    TABELA_VENDAS, quantidade=EDA(
        menor=Min,  maior=Max, contagem=Count
    )
)

In [6]:
def mostra_dados(query: Select):
  print(query)
  dados = duckdb.sql( str(query) )
  return dados.to_df()

### Vamos ver como está ficando o SQL?🤔

In [7]:
mostra_dados(query)

SELECT
	Min(v.quantidade) as menor,
	Max(v.quantidade) as maior,
	Count(v.quantidade) as contagem
FROM
	'sample_data/vendas.csv' v


,menor,maior,contagem
0,1,200,1500


### Que tal agrupar por produto?

In [8]:
from sql_blocks import GroupBy

query.add_fields('produto_id', GroupBy)
mostra_dados(query)

SELECT
	Min(v.quantidade) as menor,
	Max(v.quantidade) as maior,
	Count(v.quantidade) as contagem,
	v.produto_id
FROM
	'sample_data/vendas.csv' v
GROUP BY
	produto_id


,menor,maior,contagem,produto_id
0,3,17,13,110
1,1,19,14,79
2,5,19,6,93
3,2,57,13,62
4,3,99,10,117
...,...,...,...,...
195,7,76,5,57
196,6,20,5,138
197,1,175,3,169
198,11,15,2,56


### Uma análise diferente: Quais clientes compraram acima da média?

In [9]:
from sql_blocks import Compare, Where

query = Select(
    TABELA_VENDAS,
    quantidade=Compare.avg( Where.gt ) # gt = maior que, avg = média
).add_fields('cliente_id, dt_venda')

mostra_dados(query)

SELECT
	v.quantidade,
	v.cliente_id,
	v.dt_venda
FROM
	'sample_data/vendas.csv' v
WHERE
	v.quantidade > (SELECT Avg(v2.quantidade) FROM 'sample_data/vendas.csv' v2)


,quantidade,cliente_id,dt_venda
0,107,50,2026-03-26
1,105,70,2024-03-26
2,20,70,2025-03-31
3,62,80,2024-07-09
4,28,100,2023-07-29
...,...,...,...
204,20,72,2026-01-22
205,20,4,2023-08-27
206,70,30,2023-09-18
207,20,15,2025-05-29


### Poderia ser só o ano de 2025? Sim!

In [10]:
from sql_blocks import Year

query(
    dt_venda=Year.eq(2025)
)

mostra_dados(query)

SELECT
	v.quantidade,
	v.cliente_id,
	v.dt_venda
FROM
	'sample_data/vendas.csv' v
WHERE
	v.quantidade > (SELECT Avg(v2.quantidade) FROM 'sample_data/vendas.csv' v2) AND 
	Year(v.dt_venda) = 2025


,quantidade,cliente_id,dt_venda
0,20,70,2025-03-31
1,160,30,2025-04-19
2,154,70,2025-09-22
3,192,10,2025-12-08
4,20,65,2025-09-08
...,...,...,...
61,175,90,2025-07-19
62,31,10,2025-10-29
63,162,50,2025-06-27
64,36,50,2025-05-09


### Também dá para mostrar as maiores quantidades primeiro!

In [11]:
from sql_blocks import OrderBy

query(
    quantidade=OrderBy.DESC
)

mostra_dados(query)

SELECT
	v.quantidade,
	v.cliente_id,
	v.dt_venda
FROM
	'sample_data/vendas.csv' v
WHERE
	v.quantidade > (SELECT Avg(v2.quantidade) FROM 'sample_data/vendas.csv' v2) AND 
	Year(v.dt_venda) = 2025
ORDER BY
	v.quantidade DESC


,quantidade,cliente_id,dt_venda
0,200,80,2025-10-21
1,194,100,2025-08-03
2,192,10,2025-12-08
3,186,60,2025-01-15
4,185,20,2025-11-21
...,...,...,...
61,20,41,2025-04-01
62,20,91,2025-03-09
63,20,59,2025-02-15
64,20,27,2025-06-08


### Que tal fazer uma CTE com isso?

In [12]:
from sql_blocks import CTE, Sum, Partition

# --- Define OUTRO objeto, baseado na query (SEM mexer na query): ---
cte = CTE('melhores_clientes', [query])

# -------- O objeto cte também pode ser editado e gerar SQL: --------
cte(
    dt_ref=Year().As('ano'),
    quantidade=Sum().As('total').over(
        cliente_id=Partition, ano=OrderBy
    )
).add_fields('cliente_id').limit(50)

print(cte)
query.break_lines = True # volta a mostrar no formato anterior

WITH melhores_clientes AS (
    SELECT v.quantidade, v.cliente_id
    , v.dt_venda FROM 'sample_data/vendas.csv' v 
    WHERE v.quantidade > (SELECT Avg(v2.quantidade) 
    FROM 'sample_data/vendas.csv' v2) 
    AND  Year(v.dt_venda) = 2025 ORDER BY
     v.quantidade DESC
)SELECT Year(mc.dt_ref) as ano, Sum(mc.quantidade) OVER(
		PARTITION BY cliente_id
		ORDER BY ano
	) as total, mc.cliente_id FROM melhores_clientes mc LIMIT 50


### Só está faltando mostrar o _nome do cliente_.

In [13]:
from sql_blocks import PrimaryKey, Field

query(
    cliente_id=Select(TABELA_CLIENTES, id=PrimaryKey, nome=Field)
)

mostra_dados(query)

SELECT
	v.quantidade,
	v.cliente_id,
	v.dt_venda,
	c.nome
FROM
	'sample_data/vendas.csv' v
	JOIN 'sample_data/clientes.csv' c ON (v.cliente_id = c.id)
WHERE
	v.quantidade > (SELECT Avg(v2.quantidade) FROM 'sample_data/vendas.csv' v2) AND 
	Year(v.dt_venda) = 2025
ORDER BY
	v.quantidade DESC


,quantidade,cliente_id,dt_venda,nome
0,200,80,2025-10-21,Cliente 80
1,194,100,2025-08-03,Cliente 100
2,192,10,2025-12-08,Cliente 10
3,186,60,2025-01-15,Cliente 60
4,185,20,2025-11-21,Cliente 20
...,...,...,...,...
61,20,15,2025-01-13,Cliente 15
62,20,27,2025-01-30,Cliente 27
63,20,65,2025-09-08,Cliente 65
64,20,96,2025-02-20,Cliente 96


### Você pode remover um campo de qualquer parte da consulta

In [14]:
query.delete('quantidade',    [Where])
#                               ^^^
#                                |
#                                |
#   Se não fosse especificado, --+
#    removeria de toda parte
#   ■■■■■■■■■■■■■■■■■■■■■■■■■

mostra_dados(query)

SELECT
	v.quantidade,
	v.cliente_id,
	v.dt_venda,
	c.nome
FROM
	'sample_data/vendas.csv' v
	JOIN 'sample_data/clientes.csv' c ON (v.cliente_id = c.id)
WHERE
	Year(v.dt_venda) = 2025
ORDER BY
	v.quantidade DESC


,quantidade,cliente_id,dt_venda,nome
0,200,80,2025-10-21,Cliente 80
1,194,100,2025-08-03,Cliente 100
2,192,10,2025-12-08,Cliente 10
3,186,60,2025-01-15,Cliente 60
4,185,20,2025-11-21,Cliente 20
...,...,...,...,...
450,1,33,2025-06-06,Cliente 33
451,1,86,2025-09-17,Cliente 86
452,1,16,2025-05-23,Cliente 16
453,1,79,2025-02-06,Cliente 79


Não é recomendável usar YEAR no WHERE, então você pode `otimizar` sua consulta:

In [15]:
mostra_dados(query)

SELECT
	v.quantidade,
	v.cliente_id,
	v.dt_venda,
	c.nome
FROM
	'sample_data/vendas.csv' v
	JOIN 'sample_data/clientes.csv' c ON (v.cliente_id = c.id)
WHERE
	Year(v.dt_venda) = 2025
ORDER BY
	v.quantidade DESC


,quantidade,cliente_id,dt_venda,nome
0,200,80,2025-10-21,Cliente 80
1,194,100,2025-08-03,Cliente 100
2,192,10,2025-12-08,Cliente 10
3,186,60,2025-01-15,Cliente 60
4,185,20,2025-11-21,Cliente 20
...,...,...,...,...
450,1,33,2025-06-06,Cliente 33
451,1,86,2025-09-17,Cliente 86
452,1,16,2025-05-23,Cliente 16
453,1,79,2025-02-06,Cliente 79


### E se precisar converter para Pandas? Ou Polars? Ou MongoDB...?

In [16]:
# from sql_blocks import PandasLanguage

print(
    query.translate_to('pandas')
)

import pandas as pd

df_vendas = pd.read_csv('sample_data/vendas.csv')
df_clientes = pd.read_csv('JOIN 'sample_data/clientes.csv')
df = df_vendas

df = df[
	(df['dt_venda'] == 2025)
][[
	'quantidade',
	'cliente_id',
	'dt_venda',
	'nome'
]].sort_values(
	'quantidade',
	ascending = False
)


### Medindo variação
> Lidando com fórmulas nas consultas

In [17]:
from sql_blocks import Lag, Case, ExpressionField

query = Select(
    TABELA_VENDAS, ano=Year('dt_venda'),
    produto_id=[
      Select(
          TABELA_PRODUTOS, id=PrimaryKey,
          preco=ExpressionField('ROUND(v.quantidade * %, 2) AS faturamento')
      ),
      Field
    ],
    variacao=Lag('ROUND((faturamento - anterior) / anterior, 2)').over(
        produto_id=Partition, ano=OrderBy
    ),
    situacao=Case('variacao'
    ).when( Where.lte(-0.5)).then('crítica'
    ).when( Where.lt(0)).then('caindo'
    ).when( Where.gte(0.1)).then('subindo'
    ).else_value('estavel')
)

mostra_dados(query)

SELECT
	Year(v.dt_venda) as ano,
	ROUND(v.quantidade * p.preco, 2) AS faturamento,
	v.produto_id,
	Lag(faturamento) OVER(
		PARTITION BY produto_id
		ORDER BY ano
	) as anterior,
	ROUND((faturamento - anterior) / anterior, 2) AS variacao,
	CASE
            WHEN variacao <= -0.5 THEN 'crítica'
            WHEN variacao < 0 THEN 'caindo'
            WHEN variacao >= 0.1 THEN 'subindo'
            ELSE 'estavel'
        END AS situacao
FROM
	'sample_data/vendas.csv' v
	JOIN 'sample_data/produtos.csv' p ON (v.produto_id = p.id)


,ano,faturamento,produto_id,anterior,variacao,situacao
0,2024,767.15,15,NaN,NaN,estavel
1,2024,1534.30,15,767.15,1.00,subindo
2,2024,2761.74,15,1534.30,0.80,subindo
3,2024,153.43,15,2761.74,-0.94,crítica
4,2025,1994.59,15,153.43,12.00,subindo
...,...,...,...,...,...,...
1495,2024,1025.20,194,1845.36,-0.44,caindo
1496,2025,1025.20,194,1025.20,0.00,estavel
1497,2025,39777.76,194,1025.20,37.80,subindo
1498,2025,2665.52,194,39777.76,-0.93,crítica


### Hipótese para queda de vendas:
O estoque não está acompanhando a demanda

In [18]:
from sql_blocks import Round, Avg, RuleDateFuncReplace

query = Select(
    TABELA_VENDAS, quantidade=Round(
        Avg(), 2
    ).As('demanda', OrderBy.DESC),
    produto_id=[
        Select(
            TABELA_PRODUTOS, id=PrimaryKey,
            categoria=[Field, GroupBy],
            estoque=Sum().As('total_estoque')
        ),
        GroupBy, Field
    ],
    dt_venda=Year.eq(2025)
)

cte = CTE(
    'Produtos_indisponiveis',
     [query.optimize([RuleDateFuncReplace])]
)
cte(
    total_estoque=ExpressionField('ROUND(% - demanda*2, 2) AS disponivel'),
    demanda=Field, disponivel=Where.lt(0), produto_id=Field,
    categoria=Field
)

TABELA_INDISPONIVEIS = "'sample_data/Indisponiveis.csv'"
cmd_copy = f"COPY ({cte}) TO {TABELA_INDISPONIVEIS} (HEADER, DELIMITER ',')"
print(cmd_copy)
print('░'*50)

con = duckdb.connect()
con.execute(cmd_copy)

query = Select(TABELA_INDISPONIVEIS+' ind')

mostra_dados(query)

COPY (WITH Produtos_indisponiveis AS (
    SELECT Round(Avg(v.quantidade)
    , 2) as demanda, p.categoria, Sum(p.estoque) as total_estoque
    , v.produto_id FROM 'sample_data/vendas.csv' v 
    JOIN 'sample_data/produtos.csv' p ON (v.produto_id = p.id) 
    WHERE v.dt_venda >= '2025-01-01' 
    AND v.dt_venda <= '2025-12-31' 
    GROUP BY p.categoria, produto_id 
    ORDER BY demanda DESC
)SELECT
	ROUND(pi.total_estoque - demanda*2, 2) AS disponivel,
	pi.demanda,
	pi.produto_id,
	pi.categoria
FROM
	Produtos_indisponiveis pi
WHERE
	disponivel < 0) TO 'sample_data/Indisponiveis.csv' (HEADER, DELIMITER ',')
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
SELECT
	*
FROM
	'sample_data/Indisponiveis.csv' ind


,disponivel,demanda,produto_id,categoria
0,-158.00,192.00,94,Casa
1,-5.00,180.00,18,Beleza
2,-48.34,70.67,6,Eletrônicos
3,-61.00,69.00,112,Casa
4,-23.00,13.50,118,Eletrônicos


### O que acontece se um cliente procurar iPhone e não encontrar?🤔
Não seria melhor recomendar produtos da categoria `celular`❓?

In [19]:
from sql_blocks import Not, NamedField, Insert
Where.quoted_result = False

query = Select(
    TABELA_INDISPONIVEIS,
    categoria=Select(
      TABELA_PRODUTOS, categoria=PrimaryKey,
      id=NamedField('id_substituto')
    ),
    produto_id=[ Not.eq('p.id'), Field],
)
Where.quoted_result = True

mostra_dados(query)

SELECT
	p.id as id_substituto,
	ind.produto_id
FROM
	'sample_data/Indisponiveis.csv' ind
	JOIN 'sample_data/produtos.csv' p ON (ind.categoria = p.categoria)
WHERE
	ind.produto_id <> p.id


,id_substituto,produto_id
0,1,118
1,5,18
2,6,118
3,7,18
4,8,112
...,...,...
163,192,6
164,195,6
165,196,94
166,197,94


### Cuidando da qualidade
Selecionar somente produtos com avaliação maior que 4.5

In [20]:
from sql_blocks import SelectIN, Having

TABELA_AVALIACOES = "'sample_data/avaliacoes.csv' a"

query(
    produto_id=SelectIN(
        TABELA_AVALIACOES,
        produto_id=[Field, GroupBy],
        nota=Having.avg( Where.gt(3) )
    )
)

mostra_dados(query)

SELECT
	p.id as id_substituto,
	ind.produto_id
FROM
	'sample_data/Indisponiveis.csv' ind
	JOIN 'sample_data/produtos.csv' p ON (ind.categoria = p.categoria)
WHERE
	ind.produto_id <> p.id AND 
	ind.produto_id IN (SELECT a.produto_id FROM 'sample_data/avaliacoes.csv' a 
    GROUP BY a.produto_id HAVING Avg(a.nota) > 3)


,id_substituto,produto_id
0,1,6
1,5,18
2,7,18
3,8,94
4,10,6
...,...,...
128,184,112
129,186,112
130,196,112
131,197,112


### Preferências dos clientes
Quanto cada cliente compra de cada categoria
> Fazendo PIVOT nas categorias...

In [21]:
from sql_blocks import Pivot

query = Select(
    TABELA_VENDAS, cliente_id=[Field, GroupBy],
    produto_id=Select(
        TABELA_PRODUTOS, id=PrimaryKey,
        categoria=Pivot(
            'v.quantidade',
            'Casa Beleza Eletrônicos Esport Roupas Alimentos'.split()
        )
    )
)

mostra_dados(query)

SELECT
	v.cliente_id,
	Sum(CASE WHEN p.categoria = 'Casa' THEN v.quantidade ELSE 0 END) as Casa,
	Sum(CASE WHEN p.categoria = 'Alimentos' THEN v.quantidade ELSE 0 END) as Alimentos,
	Sum(CASE WHEN p.categoria = 'Eletrônicos' THEN v.quantidade ELSE 0 END) as Eletrônicos,
	Sum(CASE WHEN p.categoria = 'Esport' THEN v.quantidade ELSE 0 END) as Esport,
	Sum(CASE WHEN p.categoria = 'Beleza' THEN v.quantidade ELSE 0 END) as Beleza,
	Sum(CASE WHEN p.categoria = 'Roupas' THEN v.quantidade ELSE 0 END) as Roupas
FROM
	'sample_data/vendas.csv' v
	JOIN 'sample_data/produtos.csv' p ON (v.produto_id = p.id)
GROUP BY
	v.cliente_id


,cliente_id,Casa,Alimentos,Eletrônicos,Esport,Beleza,Roupas
0,94,57.0,28.0,32.0,0.0,39.0,27.0
1,25,44.0,10.0,15.0,0.0,44.0,31.0
2,38,31.0,18.0,33.0,0.0,72.0,2.0
3,56,1.0,36.0,46.0,0.0,6.0,28.0
4,21,40.0,30.0,41.0,0.0,23.0,38.0
...,...,...,...,...,...,...,...
95,92,23.0,34.0,66.0,0.0,46.0,0.0
96,5,0.0,11.0,98.0,0.0,0.0,0.0
97,91,27.0,0.0,34.0,0.0,35.0,33.0
98,74,7.0,37.0,9.0,0.0,19.0,19.0
